# GridPulse — Phase 1 Walkthrough

This notebook establishes the first analytical layer for the project using the deterministic development fixture. Replace the fixture with an authenticated EIA pull before interpreting values as real grid behavior.


## 1. Build an hourly operating table

We start with a reproducible fixture so feature engineering, QA, and visual logic can be tested before an API key is configured. The production path uses EIA-930 `D`, `DF`, `NG`, and `TI` series.


In [ ]:
from gridpulse.demo import make_demo_data
from gridpulse.features import add_operational_features
from gridpulse.stress import add_stress_score

# Create enough hourly history to expose daily and weekly load-shape behavior.
raw = make_demo_data(hours=24 * 30)
df = add_stress_score(add_operational_features(raw))
df[["period", "demand_mwh", "forecast_mwh", "forecast_error_mwh", "demand_ramp_pct", "stress_score"]].head()


### What this output tells us

The engineered table keeps the original operating signals alongside derived quantities. That matters because every stress flag can be traced back to a demand level, forecast miss, ramp, and interchange condition rather than appearing as an unexplained model score.


## 2. Measure forecast error

A day-ahead forecast can look acceptable on average while still missing the specific high-demand hours that matter operationally, so the project keeps both signed and absolute error.


In [ ]:
forecast_summary = {
    "mae_mwh": df["abs_forecast_error_mwh"].mean(),
    "median_abs_error_mwh": df["abs_forecast_error_mwh"].median(),
    "mean_signed_error_mwh": df["forecast_error_mwh"].mean(),
}
forecast_summary


### What this output tells us

MAE summarizes the typical miss, while the signed mean helps reveal systematic under- or over-forecasting. In the real EIA analysis we will also slice these errors by season, hour of day, demand percentile, and peak events instead of relying on one aggregate number.


## 3. Inspect the highest-stress hours

The screening score is useful only if the reviewer can immediately see the evidence behind a flag. We therefore sort the highest-scoring hours and retain the component signals.


In [ ]:
event_columns = [
    "period", "demand_mwh", "abs_forecast_error_mwh",
    "demand_ramp_pct", "interchange_share", "stress_score", "stress_band"
]
df.nlargest(10, "stress_score")[event_columns]


### What this output tells us

A high screening score should represent several unusual conditions occurring together, not merely a high load hour. This event table is the audit trail for that interpretation. The score is not a blackout probability or reliability standard.


## 4. Next analytical step

Once the EIA API key is configured, the same cells will run on real hourly data. The next phase adds missing-hour QA, generation-by-fuel analysis, seasonal-naive forecasting, rolling-origin validation, and README figures generated from real EIA outputs.
